# Dataset postprocessing

Turns the raw output of `bash_scripts/00_bias_injection.sh` into the final dataset:

1. Diversifies repetitive trigger phrasings (authority, herding, overconfidence, framing). Each matching sentence is rewritten with probability 0.7.
2. Adds the 150 unbiased base sentences (`biases = "none"`) and shuffles.

Bias injection is sampled, so rerunning it will not reproduce `data/triggerbank.csv` exactly. The result is written to `outputs/` so the released dataset is never overwritten.

In [ ]:
import random
import re

import pandas as pd

INJECTION_OUTPUT = "../outputs/bias_injection/biased_sentences_gpt-5.4_injection_rewrite.csv"
BASE_SENTENCES = "../data/base_sentences.csv"
OUTPUT_PATH = "../outputs/triggerbank_regenerated.csv"

random.seed(42)

# Probability of replacement (70% replace, 30% keep original)
REPLACEMENT_PROBABILITY = 0.7

In [ ]:
# Authority: vary BOTH phrase and source
AUTHORITY_PHRASES = [
    "according to",
    "as reported by", 
    "per data from",
    "cited by",
    "in figures compiled by",
    "as noted by",
    "tracked by",
]

AUTHORITY_SOURCES = [
    "Reuters",
    "Bloomberg", 
    "Goldman Sachs",
    "JP Morgan",
    "Morgan Stanley",
    "UBS",
    "Barclays",
    "Moody's",
    "S&P",
    "Fitch",
    "Financial Times",
]

# Herding
HERDING_FIND = [
    "as market participants adjusted exposure",
    "amid broader institutional repositioning",
    "following wider portfolio adjustments in the sector",
    "with market participants adjusting their exposure",
]
HERDING_REPLACEMENTS = [
    "as peers repositioned",
    "amid wider portfolio shifts",
    "as institutions reallocated",
    "with funds adjusting positions",
    "amid parallel moves by investors",
    "as peers adjusted holdings",
    "with institutions shifting exposure",
    "amid sector-wide repositioning",
    "as portfolios shifted",
    "with peers rebalancing",
]

# Overconfidence
OVERCONFIDENCE_FIND = [
    "a pattern showing no signs of changing",
    "showing no signs of changing",
    "expected to hold",
    "a pattern expected to hold",
    "a level expected to hold",
    "a level showing no signs of changing",
    "set to remain",
    "poised to continue",
]
OVERCONFIDENCE_REPLACEMENTS = [
    "likely to persist",
    "appearing entrenched",
    "unlikely to shift",
    "showing persistence",
    "appearing stable",
    "likely to hold",
    "seeming fixed",
    "appearing set",
    "unlikely to change",
    "showing stability",
]


def replace_authority(text):
    """Replace authority phrases with probability."""
    if random.random() > REPLACEMENT_PROBABILITY:
        return text
    
    sources = AUTHORITY_SOURCES
    phrases = ["according to", "as reported by", "per", "as noted by", 
               "tracked by", "cited by", "in figures compiled by", "in data from"]
    
    for source in sources:
        for phrase in phrases:
            pattern = f"{phrase} {source}"
            if pattern.lower() in text.lower():
                new_phrase = random.choice(AUTHORITY_PHRASES)
                new_source = random.choice(AUTHORITY_SOURCES)
                text = re.sub(re.escape(pattern), f"{new_phrase} {new_source}", text, flags=re.IGNORECASE, count=1)
                return text
    return text


def replace_herding(text):
    """Replace herding phrases with probability."""
    if random.random() > REPLACEMENT_PROBABILITY:
        return text
    
    for phrase in HERDING_FIND:
        if phrase.lower() in text.lower():
            replacement = random.choice(HERDING_REPLACEMENTS)
            text = re.sub(re.escape(phrase), replacement, text, flags=re.IGNORECASE, count=1)
            return text
    return text


def replace_overconfidence(text):
    """Replace overconfidence phrases with probability."""
    if random.random() > REPLACEMENT_PROBABILITY:
        return text
    
    for phrase in OVERCONFIDENCE_FIND:
        if phrase.lower() in text.lower():
            replacement = random.choice(OVERCONFIDENCE_REPLACEMENTS)
            text = re.sub(re.escape(phrase), replacement, text, flags=re.IGNORECASE, count=1)
            return text
    return text


def replace_framing(text):
    """Replace framing words with probability."""
    if random.random() > REPLACEMENT_PROBABILITY:
        return text
    
    replacements = {
        "a modest": ["a slim", "a limited", "a small"],
        "a mere": ["a slim", "a bare", "a small"],
        "a solid": ["a stable", "a steady", "a maintained"],
        "a still-solid": ["a stable", "a steady", "a held"],
    }
    
    for find, options in replacements.items():
        if find in text.lower():
            replacement = random.choice(options)
            text = re.sub(re.escape(find), replacement, text, flags=re.IGNORECASE, count=1)
            return text
    return text

In [ ]:
# Diversify trigger phrasings
df = pd.read_csv(INJECTION_OUTPUT)
df = df[["original_sentence", "label", "biased_sentence", "biases"]]

df["biased_sentence"] = df["biased_sentence"].apply(replace_authority)
df["biased_sentence"] = df["biased_sentence"].apply(replace_herding)
df["biased_sentence"] = df["biased_sentence"].apply(replace_overconfidence)
df["biased_sentence"] = df["biased_sentence"].apply(replace_framing)

In [ ]:
# Add the unbiased base sentences
original_df = pd.read_csv(BASE_SENTENCES)

original_df = original_df.rename(columns={"sentence": "original_sentence"})
original_df["biased_sentence"] = original_df["original_sentence"]  # Same text, no bias
original_df["biases"] = "none"

# Keep only columns that match
columns_to_keep = ["original_sentence", "biased_sentence", "label", "biases"]
df = df[columns_to_keep]
original_df = original_df[columns_to_keep]

# Combine and shuffle
combined_df = pd.concat([df, original_df], ignore_index=True)
combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Biased examples: {len(df)}")
print(f"Unbiased examples: {len(original_df)}")
print(f"Total: {len(combined_df)}")
print(f"\nBias distribution:\n{combined_df['biases'].value_counts()}")

combined_df.to_csv(OUTPUT_PATH, index=False)
print(f"\nSaved to: {OUTPUT_PATH}")